# 🤖 Sesiunea 23 · Transformere în practică

**Azi folosim modele gata antrenate** pentru sarcini de limbaj, fiecare cu **2 rânduri de cod**:

| Sarcina | Ce face | Tip de Transformer |
|---|---|---|
| Sentiment | pozitiv / negativ / neutru | encoder |
| Zero-shot | pune textul într-o categorie aleasă de tine | encoder |
| NER | găsește nume, locuri, sume, date | encoder |
| Întrebare-răspuns | găsește răspunsul într-un text | encoder |
| Traducere | română → engleză | encoder-decoder |
| Generare | scrie text nou | decoder |

Rețeta e mereu aceeași: **`pipeline(sarcina, model)`** → îi dai text → primești rezultatul.

## Pregătire (rulează o dată)

In [26]:
import pandas as pd
from transformers import pipeline, AutoTokenizer

Câteva recenzii reale în română (același set ca la sesiunile de NLP și embeddings). Luăm 6 scurte, la întâmplare.

In [27]:
URL = "https://raw.githubusercontent.com/dumitrescustefan/Romanian-Transformers/examples/examples/sentiment_analysis/ro/train.csv"
toate = pd.read_csv(URL)["text"].dropna()
recenzii = toate[toate.str.len().between(40, 200)].sample(6, random_state=1).tolist()
recenzii

['excelent\nmerita toti banii, cam asemanator cu modelul 6.',
 'excelent\noricine ce ar zice, pentru o aplicatie in automatizari este perfect, mai ieftin decat rasberry pi cu ecran tactil',
 'un super film mi-a placut foarte mult si l-am vazut de vreo 10 ori dar totusi nu-i dau 5 stele pentru ca eu cred ca terminator 2 este mai bun decat titanic.',
 'un film foarte bun, cel mai mult mi-a placut de robert downey jr cu replicile sale si de scarlett johansson :d',
 'ii dau nota 10 pentru ca se vede in totalitate investitia facuta in acest film.este intr-adevar, grandios!',
 'capodopera de film, nam cuvinte, isi merita pe deplin toate premiile obtinute.']

## 1 · Sentiment: pozitiv, negativ sau neutru? 🙂🙁
*(după slide-urile „pipeline” și „Sarcini NLP”)*

**Pasul 1:** încărcăm un model gata antrenat (un encoder, rudă cu XLM-RoBERTa de data trecută).

In [28]:
sentiment = pipeline("sentiment-analysis", model="cardiffnlp/twitter-xlm-roberta-base-sentiment", use_fast=False)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

**Pasul 2:** îi dăm o propoziție.

In [29]:
sentiment("Livrarea a fost rapida, sunt foarte multumit!")

[{'label': 'positive', 'score': 0.9328766465187073}]

**Pasul 3:** îi dăm toate recenziile deodată și le punem într-un tabel.

In [30]:
tabel = pd.DataFrame(sentiment(recenzii, truncation=True))
tabel["recenzie"] = recenzii
tabel

,label,score,recenzie
0,positive,0.834861,"excelent\nmerita toti banii, cam asemanator cu..."
1,positive,0.775018,"excelent\noricine ce ar zice, pentru o aplicat..."
2,positive,0.901752,un super film mi-a placut foarte mult si l-am ...
3,positive,0.866445,"un film foarte bun, cel mai mult mi-a placut d..."
4,positive,0.935656,ii dau nota 10 pentru ca se vede in totalitate...
5,positive,0.875121,"capodopera de film, nam cuvinte, isi merita pe..."


**🤔 Întrebare** · Modelul a fost antrenat pe tweet-uri, în 8 limbi, iar româna **nu** e printre ele. Cum de merge totuși? Unde greșește?

> ✍️ *Răspunsul tău:* ....

## 2 · Zero-shot: categoriile le alegi tu 🗂️
*(după slide-ul „Zero-shot”)*

„Zero-shot” = modelul **n-a văzut niciun exemplu** cu categoriile noastre. Îi dăm doar numele lor.

In [31]:
zero_shot = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli", use_fast=False)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

In [32]:
categorii = ["livrare", "calitatea produsului", "pret", "film"]
zero_shot("Coletul a ajuns dupa doua saptamani.", candidate_labels=categorii)

{'sequence': 'Coletul a ajuns dupa doua saptamani.',
 'labels': ['livrare', 'pret', 'film', 'calitatea produsului'],
 'scores': [0.972188413143158,
  0.013516548089683056,
  0.00808095745742321,
  0.006214065011590719]}

Pentru fiecare recenzie păstrăm doar categoria câștigătoare:

In [33]:
tabel["categorie"] = [zero_shot(r, candidate_labels=categorii)["labels"][0] for r in recenzii]
tabel[["recenzie", "categorie"]]

,recenzie,categorie
0,"excelent\nmerita toti banii, cam asemanator cu...",pret
1,"excelent\noricine ce ar zice, pentru o aplicat...",pret
2,un super film mi-a placut foarte mult si l-am ...,film
3,"un film foarte bun, cel mai mult mi-a placut d...",calitatea produsului
4,ii dau nota 10 pentru ca se vede in totalitate...,calitatea produsului
5,"capodopera de film, nam cuvinte, isi merita pe...",film


**Exercițiul 2.1 · Categoriile tale**

Schimbă lista `categorii` (ex: „baterie”, „ecran”, „sunet”, „curier”) și rulează din nou ultimele două celule. Ce se întâmplă dacă pui o categorie care nu se potrivește cu nicio recenzie?

> ✍️ *Răspunsul tău:* ....

In [34]:
categorii = ["baterie", "ecran", "sunet", "curier"]  # ✏️ lista ta de categorii
zero_shot(recenzii[0], candidate_labels=categorii)

{'sequence': 'excelent\nmerita toti banii, cam asemanator cu modelul 6.',
 'labels': ['sunet', 'baterie', 'ecran', 'curier'],
 'scores': [0.29697471857070923,
  0.2638770043849945,
  0.26233533024787903,
  0.17681294679641724]}

## 3 · NER: nume, locuri, sume, date 🏷️
*(după slide-ul „NER”)*

Un model antrenat special pe **limba română** (RONEC, un set de date românesc).

In [ ]:
ner = pipeline("ner", model="dumitrescustefan/bert-base-romanian-ner", aggregation_strategy="simple", use_fast=False)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [39]:
text = "Ana a cumpărat de la eMAG un telefon de 2000 de lei, livrat în Cluj pe 12 martie."
pd.DataFrame(ner(text))[["entity_group", "word", "score"]]

,entity_group,word,score
0,LABEL_1,Ana,0.997072
1,LABEL_0,a cumpărat de la,0.999869
2,LABEL_3,eMAG,0.986399
3,LABEL_0,un telefon de,0.999760
4,LABEL_21,2000,0.997427
5,LABEL_22,de lei,0.999176
6,LABEL_0,", livrat în",0.999887
7,LABEL_5,Cluj,0.997874
8,LABEL_0,pe,0.998237
9,LABEL_17,12,0.997414


**🤔 Întrebare** · Ce etichete a găsit? Ce ar putea face un magazin online cu așa ceva, pe mii de mesaje de la clienți?

> ✍️ *Răspunsul tău:* ....

In [40]:
ner("Maria Ionescu din Iasi a sunat ieri la firma Altex.")  # ✏️ propoziția ta

[{'entity_group': 'LABEL_1',
  'score': np.float32(0.9985416),
  'word': 'Maria',
  'start': 0,
  'end': 5},
 {'entity_group': 'LABEL_2',
  'score': np.float32(0.9990728),
  'word': 'Ionescu',
  'start': 6,
  'end': 13},
 {'entity_group': 'LABEL_0',
  'score': np.float32(0.9984787),
  'word': 'din',
  'start': 14,
  'end': 17},
 {'entity_group': 'LABEL_5',
  'score': np.float32(0.9943375),
  'word': 'Iasi',
  'start': 18,
  'end': 22},
 {'entity_group': 'LABEL_0',
  'score': np.float32(0.99981457),
  'word': 'a sunat',
  'start': 23,
  'end': 30},
 {'entity_group': 'LABEL_17',
  'score': np.float32(0.993138),
  'word': 'ieri',
  'start': 31,
  'end': 35},
 {'entity_group': 'LABEL_0',
  'score': np.float32(0.99690706),
  'word': 'la firma',
  'start': 36,
  'end': 44},
 {'entity_group': 'LABEL_3',
  'score': np.float32(0.97921515),
  'word': 'Alte',
  'start': 45,
  'end': 49},
 {'entity_group': 'LABEL_4',
  'score': np.float32(0.9947941),
  'word': '##x',
  'start': 49,
  'end': 50},
 

## 4 · Traducere: encoder + decoder 🌍
*(după slide-ul „Encoder-decoder”)*

Un model de traducere are **două părți**: encoder-ul citește propoziția în română, decoder-ul scrie traducerea în engleză, cuvânt cu cuvânt.

In [46]:
from transformers import MarianTokenizer, MarianMTModel

tok = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-roa-en")
model = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-roa-en")

def traducere(text):
    out = model.generate(**tok(text, return_tensors="pt"))
    return tok.decode(out[0], skip_special_tokens=True)

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

In [52]:
print(traducere("Timișoara este un oraș frumos."))

Timisoara is a beautiful city.


In [48]:
traducere("Coletul a ajuns repede, dar cutia era deteriorată.")

'The collection arrived quickly, but the box was damaged.'

**🤔 Întrebare** · Traducerea e bună? Ce se întâmplă cu recenziile fără diacritice sau cu greșeli de scriere?

> ✍️ *Răspunsul tău:* ....

## 5 · Generare: decoder-ul scrie text nou ✍️
*(după slide-ul „Decoder”)*

Un model **decoder** (familia ChatGPT, Claude, Gemini), dar foarte mic: 0,5 miliarde de parametri, ca să încapă în Colab. Data viitoare lucrăm cu modele mari, prin internet.

In [53]:
chat = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", use_fast=False)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

O funcție mică: îi trimitem o cerere și primim doar textul răspunsului.

In [54]:
def scrie(cerere):
    mesaje = [{"role": "user", "content": cerere}]
    rezultat = chat(mesaje, max_new_tokens=100)[0]["generated_text"]
    # Transformers poate returna fie istoricul conversației, fie textul direct.
    if isinstance(rezultat, str):
        return rezultat
    if isinstance(rezultat, list) and rezultat:
        ultimul = rezultat[-1]
        if isinstance(ultimul, dict):
            return ultimul.get("content", ultimul.get("generated_text", str(ultimul)))
        return str(ultimul)
    return ""

In [57]:
print(scrie("Write a short, friendly slogan for a coffee shop."))

" Brew your own cup – taste the difference!"


In [59]:
print(scrie("Scrie un slogan scurt pentru o cafenea din Cluj."))

"Cluj, café în clădiri! Acesta este vreme de sănătate și felul de putere către cafe."


**🤔 Întrebare** · Compară răspunsul în engleză cu cel în română. Ce observi? De ce?

> ✍️ *Răspunsul tău:* ....

## Recap 🎓

| Sarcina | Pipeline | Tip | Ce primești |
|---|---|---|---|
| Sentiment | `sentiment-analysis` | encoder | o etichetă + scor |
| Zero-shot | `zero-shot-classification` | encoder | categoria ta cea mai potrivită |
| NER | `ner` | encoder | entități: persoane, firme, locuri, sume, date |
| Întrebare-răspuns | `question-answering` | encoder | o bucată din text |
| Traducere | `translation` | encoder-decoder | text în altă limbă |
| Generare | `text-generation` | decoder | text nou |

- Rețeta e mereu: **`pipeline(sarcina, model)`**, apoi îi dai text.
- **Encoder** = înțelege (clasifică, găsește). **Decoder** = scrie. **Encoder-decoder** = transformă un text în alt text.
- Modelele mici fac greșeli: alege modelul după **sarcină** și **limbă**.

# 🧪 Do it yourself
Alege 2–3 exerciții. Fiecare are 1–2 rânduri de cod.

**Exercițiul 1 · Recenzia ta.** Scrie o recenzie pozitivă și una ironică. Le prinde modelul de sentiment?

In [ ]:
sentiment(["Super produs, il recomand!", "Super, s-a stricat a doua zi."])      # ✏️ recenziile tale

**Exercițiul 2 · Sortează e-mailuri.** Folosește zero-shot pe 3 mesaje inventate, cu categoriile „reclamatie”, „intrebare”, „multumire”.

In [ ]:
zero_shot("Nu mi-a venit factura pentru comanda de ieri.", candidate_labels=["reclamatie", "intrebare", "multumire"])

**Exercițiul 3 · Anonimizare.** Găsește numele din textul de mai jos cu `ner` și gândește-te cum le-ai ascunde.

In [ ]:
ner("Maria Ionescu din Iasi a sunat ieri la firma Altex.")

**Exercițiul 4 · Generare.** Cere modelului mic o rețetă sau o poezie de 4 versuri. Găsești greșeli sau lucruri inventate?

In [ ]:
print(scrie("Write a 4-line poem about autumn."))

**Reflecție finală**

> ✍️ Care sarcină ți s-a părut cea mai utilă pentru meseria ta de dinainte? Ce ai automatiza cu ea?

**💡 Pentru trainer**

> ✍️ *Răspunsul tău:* ....